# Практика 2. pandas: побудова DataFrame

**Мета:** побудувати DataFrame з декількох масивів, попрацювати з індексом, `.loc`/`.iloc` і створити обчислювані стовпці.

**Варіант 1 — Київ.** Дані навчальні, наближені.

In [1]:
import pandas as pd

## Наскрізна демонстрація: магазин

In [2]:
shop = pd.DataFrame({
    "item": ["хліб", "молоко", "яблука", "цукор", "чай"],
    "price": [28, 42, 55, 38, 65],
    "stock": [120, 60, 90, 40, 25],
})
shop_by_item = shop.set_index("item")
display(shop_by_item.loc["молоко"])
shop = shop_by_item.reset_index()
shop.index = range(1, 6)
display(shop.loc[1:3, "price"])
display(shop.iloc[0:3, 1])
display(shop[(shop["price"] < 50) & (shop["stock"] > 50)])
shop["total_value"] = shop["price"] * shop["stock"]
shop["low_stock"] = shop["stock"] < 50
def stock_label(qty):
    if qty < 50:
        return "мало"
    elif qty < 100:
        return "достатньо"
    return "багато"
shop["stock_label"] = shop["stock"].apply(stock_label)
shop

price    42
stock    60
Name: молоко, dtype: int64

1    28
2    42
3    55
Name: price, dtype: int64

1    28
2    42
3    55
Name: price, dtype: int64

,item,price,stock
1,хліб,28,120
2,молоко,42,60


,item,price,stock,total_value,low_stock,stock_label
1,хліб,28,120,3360,False,багато
2,молоко,42,60,2520,False,достатньо
3,яблука,55,90,4950,False,достатньо
4,цукор,38,40,1520,True,мало
5,чай,65,25,1625,True,мало


## Крок 1. Побудова DataFrame власного варіанта

In [3]:
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
avg_temp = [3, 8, 3, 8, 13, 17, 19, 18, 13, 8, 3, 0]
precip = [45, 40, 38, 42, 55, 75, 85, 70, 60, 50, 48, 45]
df = pd.DataFrame({"month": months, "avg_temp": avg_temp, "precip": precip})
display(df.head())
print(df.shape)
display(df.dtypes)

,month,avg_temp,precip
0,Січ,3,45
1,Лют,8,40
2,Бер,3,38
3,Кві,8,42
4,Тра,13,55


(12, 3)


month         str
avg_temp    int64
precip      int64
dtype: object

## Завдання 1. Індекс

In [4]:
display(df)
print("df.index:", df.index)
df_by_month = df.set_index("month")
display(df_by_month.loc["Чер"])
df = df_by_month.reset_index()
display(df)

,month,avg_temp,precip
0,Січ,3,45
1,Лют,8,40
2,Бер,3,38
3,Кві,8,42
4,Тра,13,55
5,Чер,17,75
6,Лип,19,85
7,Сер,18,70
8,Вер,13,60
9,Жов,8,50


df.index: RangeIndex(start=0, stop=12, step=1)


avg_temp    17
precip      75
Name: Чер, dtype: int64

,month,avg_temp,precip
0,Січ,3,45
1,Лют,8,40
2,Бер,3,38
3,Кві,8,42
4,Тра,13,55
5,Чер,17,75
6,Лип,19,85
7,Сер,18,70
8,Вер,13,60
9,Жов,8,50


## Завдання 2. Відбір рядків і стовпців

`.loc` працює з підписами індексу, `.iloc` — з позиціями.

In [5]:
df.index = range(1, 13)
display(df.loc[1:6, "avg_temp"])
display(df.iloc[0:6, 1])
display(df[["month", "precip"]])

1     3
2     8
3     3
4     8
5    13
6    17
Name: avg_temp, dtype: int64

1     3
2     8
3     3
4     8
5    13
6    17
Name: avg_temp, dtype: int64

,month,precip
1,Січ,45
2,Лют,40
3,Бер,38
4,Кві,42
5,Тра,55
6,Чер,75
7,Лип,85
8,Сер,70
9,Вер,60
10,Жов,50


## Завдання 3. Фільтрація за умовою

In [6]:
display(df.loc[df["avg_temp"] > 15, ["month", "avg_temp"]])
display(df.loc[(df["avg_temp"] > 10) & (df["precip"] > 50), ["month", "avg_temp", "precip"]])
display(df.loc[(df["avg_temp"] < 0) | (df["precip"] > 80), ["month", "avg_temp", "precip"]])

,month,avg_temp
6,Чер,17
7,Лип,19
8,Сер,18


,month,avg_temp,precip
5,Тра,13,55
6,Чер,17,75
7,Лип,19,85
8,Сер,18,70
9,Вер,13,60


,month,avg_temp,precip
7,Лип,19,85


## Завдання 4. Обчислювані стовпці

In [7]:
df["avg_temp_f"] = df["avg_temp"] * 9 / 5 + 32
df["is_wet"] = df["precip"] > df["precip"].mean()
def month_season(month_number):
    if month_number in [12, 1, 2]:
        return "зима"
    if month_number in [3, 4, 5]:
        return "весна"
    if month_number in [6, 7, 8]:
        return "літо"
    return "осінь"
df["season"] = df.index.to_series().apply(month_season).to_numpy()
display(df)

,month,avg_temp,precip,avg_temp_f,is_wet,season
1,Січ,3,45,37.4,False,зима
2,Лют,8,40,46.4,False,зима
3,Бер,3,38,37.4,False,весна
4,Кві,8,42,46.4,False,весна
5,Тра,13,55,55.4,True,весна
6,Чер,17,75,62.6,True,літо
7,Лип,19,85,66.2,True,літо
8,Сер,18,70,64.4,True,літо
9,Вер,13,60,55.4,True,осінь
10,Жов,8,50,46.4,False,осінь


## Письмові відповіді та підсумок

**1.** `.loc[1:6]` використовує підписи індексу й включає праву межу, а `.iloc[0:6]` використовує позиції 0–5 і не включає праву межу. Тому кількість рядків може відрізнятися, якщо підписи та позиції не збігаються.

**2.** `and` очікує одне булеве значення, а порівняння стовпців повертає булеву Series. Для поелементного І потрібен `&` з дужками навколо умов; поелементне АБО виконується через `|`.

**3.** Векторизована формула застосовується до всього стовпця одразу й підходить для однакової математичної операції. `.apply()` викликає власну функцію для кожного значення і зручний для складної умовної логіки.

**Підсумок:** DataFrame побудовано для Києва; перевірено форму й типи; використано `.loc` та `.iloc`; виконано фільтрацію; створено стовпці формулою і через `.apply()`.